## Dataframe Schemas
See the [README.md:Dataframe Schemas](https://github.com/panoseti/panoseti/blob/adc-to-pe/adc-to-pe/README.md#dataframe-schemas) section for documentation of the dataframe formats.

In [1]:
from IPython import display
from pathlib import Path
import os
import numpy
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from rich import print
from rich.pretty import pprint
import sys
import json

# Import utils from other panoseti directories
repo_root = Path('..')
util_path = repo_root / 'util'
control_utils = repo_root / 'control/utils'
for p in [util_path, control_utils]:
    sys.path.append(str(p))

import pff, config_file, pixel_coords, util

In [2]:
quabo_config_root = repo_root / 'control/quabos'
os.listdir(quabo_config_root)

['detovervol_3v',
 'quabo_info.json',
 'detovervol_2v',
 'quabo_pixelmap_maroc2phys_bga.json',
 'quabo_pixelmap_maroc2phys_qfp.json',
 'detector_info.json',
 '.ipynb_checkpoints',
 'quabo_pixelmap_phys2maroc_qfp.json',
 'quabo_pixelmap_phys2maroc_bga.json']

In [3]:
obs_dir = Path('obs_Lick.start_2024-07-25T04:34:06Z.runtype_sci-data.pffd')
os.listdir(obs_dir)

['start_2024-07-25T04_34_46Z.dp_ph256.bpp_2.module_3.seqno_0.debug_TRUNCATED.pff',
 'recording_ended',
 'daq_config.json',
 'quabo_ph_baseline.json',
 'start_2024-07-25T04_34_46Z.dp_img16.bpp_2.module_1.seqno_0.debug_TRUNCATED.pff',
 'obs_config.json',
 'start_2024-07-25T04_34_46Z.dp_ph256.bpp_2.module_1.seqno_0.debug_TRUNCATED.pff',
 '.ipynb_checkpoints',
 'data_config.json',
 'quabo_uids.json']

In [4]:
quabo_uid_path = obs_dir/'quabo_uids.json'
# quabo_uids_df = 
# quabo_uids_df

In [5]:
with open(quabo_uid_path, 'rb') as f:
    quid_j = json.load(f)
    for d_idx, dome in enumerate(quid_j['domes']):
        for m_idx, module in enumerate(dome['modules']):
            (pd.json_normalize(module))
    # quid_j  = pd.read_json(f)
    # quid_df = pd.json_normalize(
    #     quid_df['domes'],
    #     max_level=10
    # )
pprint(quid_j)

{
│   'domes': [
│   │   {
│   │   │   'modules': [
│   │   │   │   {
│   │   │   │   │   'ip_addr': '192.168.0.4',
│   │   │   │   │   'quabos': [
│   │   │   │   │   │   {'uid': '6ae310480d05824'},
│   │   │   │   │   │   {'uid': '6ae310480d08c24'},
│   │   │   │   │   │   {'uid': '6ae310480d03022'},
│   │   │   │   │   │   {'uid': '6ae310480d07024'}
│   │   │   │   │   ]
│   │   │   │   }
│   │   │   ]
│   │   },
│   │   {
│   │   │   'modules': [
│   │   │   │   {
│   │   │   │   │   'ip_addr': '192.168.0.12',
│   │   │   │   │   'quabos': [
│   │   │   │   │   │   {'uid': '6ae310480d0a425'},
│   │   │   │   │   │   {'uid': '6a238a180e0b417'},
│   │   │   │   │   │   {'uid': '6ae310480d0c024'},
│   │   │   │   │   │   {'uid': '6a238a180e0b416'}
│   │   │   │   │   ]
│   │   │   │   }
│   │   │   ]
│   │   }
│   ]
}

In [ ]:
# Load the JSON files into dictionaries
with open('obs_config.json', 'r') as f:
    obs_config = json.load(f)
with open('quabo_uids.json', 'r') as f:
    quabo_uids = json.load(f)
with open('quabo_info-checkpoint.json', 'r') as f:
    quabo_info = json.load(f)
with open('quabo_ph_baseline.json', 'r') as f:
    quabo_ph_baseline = json.load(f)
with open('quabo_calib_37.json', 'r') as f:
    quabo_calib_37 = json.load(f)
    
# --- Generate the DataFrames from the previous step ---
DETOVERVOLTAGE = obs_config.get('detector_overvoltage', 2.0)
BOARD_SERIALNO_STR = "SN037"

# From quabo_info, SN037 corresponds to UID 6b211038060d41c
QUABO_UID_TO_TEST = "6b211038060d41c"

# NOTE: The provided quabo_ph_baseline.json does not contain an entry for
# the UID corresponding to SN037. For this example to be runnable,
# we will add a dummy entry for this UID.
if not any(q['uid'] == QUABO_UID_TO_TEST for q in quabo_ph_baseline['quabos']):
    dummy_baseline_entry = quabo_ph_baseline['quabos'][0].copy()
    dummy_baseline_entry['uid'] = QUABO_UID_TO_TEST
    quabo_ph_baseline['quabos'].append(dummy_baseline_entry)
    print(f"Added dummy baseline entry for UID {QUABO_UID_TO_TEST} to run example.\n")

detector_install_df = create_detector_install_df(quabo_info)
pixel_ph_baseline_df = create_pixel_ph_baseline_df(quabo_ph_baseline)
detector_ph_calibration_df = create_detector_ph_calibration_df(
    quabo_calib=quabo_calib_37, quabo_info=quabo_info,
    board_serialno_str=BOARD_SERIALNO_STR, detovervoltage=DETOVERVOLTAGE
)
pixel_ph_gain_delta_df = create_pixel_ph_gain_delta_df(
    quabo_calib=quabo_calib_37, quabo_info=quabo_info,
    board_serialno_str=BOARD_SERIALNO_STR, detovervoltage=DETOVERVOLTAGE
)

print(f"--- Constructing conversion matrices for Quabo UID: {QUABO_UID_TO_TEST} ---")

# Construct the B_q matrix (Baselines)
B_q = construct_b_matrix(pixel_ph_baseline_df, QUABO_UID_TO_TEST)
print(f"\nB_q (Baseline) Matrix Shape: {B_q.shape}")
print(B_q[:2, :4])

# Construct the G_q matrix (Gain Deltas)
G_q = construct_g_matrix(pixel_ph_gain_delta_df, QUABO_UID_TO_TEST)
print(f"\nG_q (Gain Delta) Matrix Shape: {G_q.shape}")
print(G_q[:2, :4])

# Construct the N_q matrix ('n' coefficients)
N_q = construct_n_matrix(detector_ph_calibration_df, QUABO_UID_TO_TEST)
print(f"\nN_q ('n' coefficient) Matrix Shape: {N_q.shape}")
print("Top-left corner (Quadrant 0):")
print(N_q[:2, :4])
print("Bottom-right corner (Quadrant 3):")
print(N_q[14:, 12:])

# Construct the M_q matrix ('m' coefficients)
M_q = construct_m_matrix(detector_ph_calibration_df, QUABO_UID_TO_TEST)
print(f"\nM_q ('m' coefficient) Matrix Shape: {M_q.shape}")
print("Top-left corner (Quadrant 0):")
print(M_q[:2, :4])
print("Bottom-right corner (Quadrant 3):")
print(M_q[14:, 12:])

# --- Demonstrate the final conversion ---
print("\n--- Demonstrating ADC to P.E. Conversion ---")
# Create a random 16x16 raw image frame (I_q)
I_q = np.random.randint(0, 500, size=(16, 16), dtype=np.float64)
print(f"Sample Raw Image (I_q) created with shape {I_q.shape}")

# Convert to Photoelectron units
pe_image = convert_adc_to_pe(I_q, B_q, G_q, N_q, M_q)
print(f"Converted P.E. Image created with shape {pe_image.shape}")
print("Sample of converted P.E. values (top-left 2x4):")
print(pe_image[:2, :4])